In [1]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
#import the data from csv fike
csv_filename= "../datacleaning-pipeline/pnw_model_dataset.csv"
df = pd.read_csv(csv_filename, dtype={'gauge_id':str}, parse_dates=['date'])

In [3]:
#seelction of the variables
input_columns = ['prcp_z', 'tmax_z', 'srad_z','vp_z', 'dayl_z', 'drainage_area_km2_z', 'elevation_m_z', 'slope_m_per_km_z','frac_forest_z', 'mean_precip_mm_day_z', 'mean_pet_mm_day_z']
target_column = ['log_runoff']

In [4]:
#Training vs Testing Period and Data
training_df=df[df['split'] =='train']
testing_df=df[df['split']=='test']

X_training = training_df[input_columns].values
X_testing = testing_df[input_columns].values

y_training =np.log1p(training_df[target_column].values.reshape(-1,1))
y_testing =np.log1p(testing_df[target_column].values.reshape(-1,1))

print(f'Training array of size : {X_training.shape}')

Training array of size : (149732, 11)


In [5]:
#Modeling

model = models.Sequential([
    layers.Input(shape=(X_training.shape[1],)),
    layers.Dense(50, activation ='relu'),
    layers.Dense(50, activation='relu'),
    layers.Dense(25,activation='relu'),
    layers.Dense(1, activation='linear')])

In [6]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 50)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │         2,550 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,451 (17.39 KB)

 Trainable params: 4,451 (17.39 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
print('\n Beginning training... \n')
model.fit(X_training, y_training, epochs=10, batch_size=64, validation_split=0.1, verbose=1)
print('\n Training complete!\n')


 Beginning training... 

Epoch 1/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 1s 245us/step - loss: 0.0424 - mae: 0.1514 - val_loss: 0.0822 - val_mae: 0.2309
Epoch 2/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 227us/step - loss: 0.0268 - mae: 0.1240 - val_loss: 0.0720 - val_mae: 0.2151
Epoch 3/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 228us/step - loss: 0.0248 - mae: 0.1187 - val_loss: 0.0749 - val_mae: 0.2170
Epoch 4/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 227us/step - loss: 0.0236 - mae: 0.1157 - val_loss: 0.0689 - val_mae: 0.2099
Epoch 5/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 228us/step - loss: 0.0228 - mae: 0.1133 - val_loss: 0.0797 - val_mae: 0.2236
Epoch 6/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 227us/step - loss: 0.0221 - mae: 0.1113 - val_loss: 0.0892 - val_mae: 0.2353
Epoch 7/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 227us/step - loss: 0.0215 - mae: 0.1099 - val_loss: 0.0835 - val_mae: 0.2315
Epoch 8/10
2106/2106 ━━━━━━━━━━━━━━━━━━━━ 0s 228us/step - loss: 0.0210 - mae: 0.1085 - val_loss: 0.0844 - val_mae: 0.229

In [8]:
test_loss, test_mae = model.evaluate(X_testing, y_testing, verbose=0)
raw_mae= np.expm1(test_mae)
y_predict_log =model.predict(X_testing)
y_true_raw= np.expm1(y_predict_log)
y_predict_raw= np.expm1(y_testing)

y_true = y_true_raw.flatten()
y_predict = y_predict_raw.flatten()

print(f'Test MSE (Log Loss):{test_loss:.4f}')
print(f'Test MAE (Log MAE):{test_mae:.4f}')
print(f'Test MAE (Raw mm/day):{raw_mae:.4f}')

2809/2809 ━━━━━━━━━━━━━━━━━━━━ 0s 110us/step
Test MSE (Log Loss):0.0321
Test MAE (Log MAE):0.1290
Test MAE (Raw mm/day):0.1377


In [9]:
nse_score = r2_score(np.expm1(y_testing), np.expm1(model.predict(X_testing)))
rmse_score =np.sqrt(mean_squared_error(y_true,y_predict))
print(f'This model has a NSE score of {nse_score:.4f}')
print(f'This model has a RMSE Score of {rmse_score:.4f}')

2809/2809 ━━━━━━━━━━━━━━━━━━━━ 0s 106us/step
This model has a NSE score of 0.7597
This model has a RMSE Score of 0.4282


This NSE seems way too high for a this type of not "optimized" hydrological model. I therefore ran a individual test to determine the actual NSE. Please see my report. 

In [12]:
test_df=df[df['split']=='test'].copy()
test_df['pred_raw']=y_predict_raw
test_df['true_raw']=y_true_raw

nse_individual =[]

for gauge, group in test_df.groupby('gauge_id'):
    basin_nse = r2_score(group['true_raw'],group['pred_raw'])
    nse_individual.append(basin_nse)

print(f'Median per basin NSE {np.mean(nse_individual):4f}')

Median per basin NSE 0.333473
